In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# CT retention lapsed for this table: enqueue a reseed and flip phase back to seeding.
# Only this table is affected; other tenants in the sweep continue (failure isolation).
src_database = widget("src_database")
src_schema = widget("src_schema")
src_table = widget("src_table")

store.enqueue_reseed(src_database, src_schema, src_table, reason="retention_miss")
print("reseed queued for", src_database, src_schema, src_table)